In [11]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [12]:
def load_data(base_path="data"):
    data = []

    for root, _, files in os.walk(base_path):
        for file in files:
            if file.lower().endswith((".csv")):
                path = os.path.join(root, file)

                try:
                    try:
                        df = pd.read_csv(path)
                    except:
                        df = pd.read_csv(path, sep=';')

                    df.columns = [col.lower().strip() for col in df.columns]

                    df.rename(columns={
                        'accelerometer_x': 'x',
                        'accelerometer_y': 'y',
                        'accelerometer_z': 'z'
                    }, inplace=True)

                    if not all(c in df.columns for c in ['x', 'y', 'z']):
                        continue

                    activity = os.path.basename(root)
                    df["activity"] = activity

                    data.append(df)

                except:
                    pass

    df = pd.concat(data, ignore_index=True)
    return df


In [13]:
df = load_data("data")

print(df.shape)
df['activity'].value_counts()

(193860, 4)


activity
running    102240
walking     55500
idle        31170
stairs       4950
Name: count, dtype: int64

In [14]:
def create_windows(df, window_size=50):
    X = []
    y = []

    for activity in df['activity'].unique():
        subset = df[df['activity'] == activity].reset_index(drop=True)

        for i in range(0, len(subset) - window_size, window_size):
            w = subset.iloc[i:i + window_size]

            mag = np.sqrt(w['x']**2 + w['y']**2 + w['z']**2)

            features = [
                w['x'].mean(), w['y'].mean(), w['z'].mean(),
                w['x'].std(), w['y'].std(), w['z'].std(),
                w['x'].max(), w['y'].max(), w['z'].max(),
                w['x'].min(), w['y'].min(), w['z'].min(),
                mag.mean(), mag.std()
            ]

            X.append(features)
            y.append(activity)

    return np.array(X), np.array(y)

In [15]:
X, y = create_windows(df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [16]:
scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

svm = SVC()
svm.fit(X_train_s, y_train)

y_pred_svm = svm.predict(X_test_s)

print("SVM:")
print(classification_report(y_test, y_pred_svm))

SVM:
              precision    recall  f1-score   support

        idle       1.00      1.00      1.00       143
     running       1.00      1.00      1.00       388
      stairs       1.00      0.74      0.85        23
     walking       0.97      1.00      0.99       221

    accuracy                           0.99       775
   macro avg       0.99      0.93      0.96       775
weighted avg       0.99      0.99      0.99       775



In [17]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest:")
print(classification_report(y_test, y_pred_rf))

Random Forest:
              precision    recall  f1-score   support

        idle       1.00      1.00      1.00       143
     running       1.00      1.00      1.00       388
      stairs       1.00      0.87      0.93        23
     walking       0.99      1.00      0.99       221

    accuracy                           1.00       775
   macro avg       1.00      0.97      0.98       775
weighted avg       1.00      1.00      1.00       775



Обидві моделі показали високі результати класифікації, однак Random Forest продемонстрував кращу узагальнюючу здатність, особливо для складного класу "stairs", де його recall (0.87) значно перевищує аналогічний показник SVM (0.74). Це свідчить про те, що Random Forest краще справляється з нелінійними залежностями та менш чутливий до схожості між класами. У свою чергу, SVM показує відмінні результати для добре відокремлених класів, але потребує більш точного налаштування для складних випадків.